<a href="https://colab.research.google.com/github/kokirecinos1-afk/Proyecto-integrador/blob/main/Proyecto_integrador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# 1. Instalar dependencias si no las tienes
!pip install "dask[dataframe]" mongomock pandas

import dask.dataframe as dd
import pandas as pd
import mongomock

print("--- 1. INGESTA Y PROCESAMIENTO PARALELO CON DASK ---")
# Cargar el archivo CSV de forma particionada con Dask
ddf = dd.read_csv('feedback_clientes.csv', blocksize="1MB")

# Limpiar espacios en los nombres de las columnas por seguridad
ddf = ddf.rename(columns={col: col.strip() for col in ddf.columns})

# Agrupar por la columna 'label' contando los registros de forma paralela
resumen_dask = ddf.groupby('label').size().compute()

print("¡Procesamiento paralelo con Dask exitoso!")
print(resumen_dask)


print("\n--- 2. PERSISTENCIA EN NoSQL (MONGODB) ---")
# Simulación local con mongomock (cumple con el requisito NoSQL sin requerir servidores externos)
client = mongomock.MongoClient()
db = client['db_feedback_proyectos']
coleccion = db['resumen_sentimientos']

# Limpiar colección previa para evitar duplicados
coleccion.delete_many({})

# Convertir el resultado de Dask a documentos JSON (diccionarios)
datos_dict = resumen_dask.reset_index(name='total_registros').to_dict(orient='records')

# Insertar de forma masiva en MongoDB
coleccion.insert_many(datos_dict)

print(f"¡{len(datos_dict)} documentos guardados en MongoDB con éxito!")


print("\n--- 3. CONSULTA DE VALIDACIÓN EN NoSQL ---")
# Ejecutar consulta compleja obligatoria ($gt)
print("🔍 Registros con más de 1,000 elementos:")
for doc in coleccion.find({"total_registros": {"$gt": 1000}}):
    print(doc)

--- 1. INGESTA Y PROCESAMIENTO PARALELO CON DASK ---
¡Procesamiento paralelo con Dask exitoso!
label
0    2000
1    2000
2    2000
3    2000
4    2000
dtype: int64

--- 2. PERSISTENCIA EN NoSQL (MONGODB) ---
¡5 documentos guardados en MongoDB con éxito!

--- 3. CONSULTA DE VALIDACIÓN EN NoSQL ---
🔍 Registros con más de 1,000 elementos:
{'label': 0, 'total_registros': 2000, '_id': ObjectId(6a732396-d4e6-11f1-9fec-0242ac1c000c)}
{'label': 1, 'total_registros': 2000, '_id': ObjectId(6a732724-d4e6-11f1-9fec-0242ac1c000c)}
{'label': 2, 'total_registros': 2000, '_id': ObjectId(6a7328fa-d4e6-11f1-9fec-0242ac1c000c)}
{'label': 3, 'total_registros': 2000, '_id': ObjectId(6a732aa8-d4e6-11f1-9fec-0242ac1c000c)}
{'label': 4, 'total_registros': 2000, '_id': ObjectId(6a732c2e-d4e6-11f1-9fec-0242ac1c000c)}


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from abc import ABC, abstractmethod

# 1. Definir la Clase Base Abstracta (Patrón Strategy exigido por el bootcamp)
class ModeloBase(ABC):
    """Contrato común para todos los modelos de clasificación de texto."""
    def __init__(self, nombre: str):
        self.nombre = nombre
        self._entrenado = False

    @abstractmethod
    def entrenar(self, X_train, y_train):
        pass

    @abstractmethod
    def predecir(self, X_test):
        pass

# 2. Implementar una clase concreta para el modelo de Machine Learning
class ModeloLogisticRegressionPOO(ModeloBase):
    def __init__(self):
        super().__init__("Regresión Logística con TF-IDF")
        # Pipeline integrado para evitar fuga de datos (Data Leakage)
        self.pipeline = Pipeline([
            ('tfidf', TfidfVectorizer(max_features=2000)), # Corregido sin stop_words='spanish'
            ('clf', LogisticRegression(max_iter=300))
        ])

    def entrenar(self, X_train, y_train):
        print(f"[{self.nombre}] Entrenando modelo...")
        self.pipeline.fit(X_train, y_train)
        self._entrenado = True
        print(f"[{self.nombre}] ¡Entrenamiento finalizado con éxito!")

    def predecir(self, X_test):
        if not self._entrenado:
            raise RuntimeError("El modelo aún no ha sido entrenado.")
        return self.pipeline.predict(X_test)


print("--- CARGANDO DATOS PARA EL MODELO ---")
df = pd.read_csv('feedback_clientes.csv')
df = df.dropna(subset=['text', 'label'])

# Separar características (X) y etiquetas (y)
X = df['text']
y = df['label']

# Partición de datos (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Instanciar y ejecutar nuestro modelo orientado a objetos
modelo_ia = ModeloLogisticRegressionPOO()
modelo_ia.entrenar(X_train, y_train)

# Predicciones y Evaluación
predicciones = modelo_ia.predecir(X_test)
acc = accuracy_score(y_test, predicciones)

print(f"\n🎯 Métricas del Modelo:")
print(f"Accuracy (Exactitud): {acc:.4f}")
print("\nReporte detallado de clasificación:")
print(classification_report(y_test, predicciones))

--- CARGANDO DATOS PARA EL MODELO ---
[Regresión Logística con TF-IDF] Entrenando modelo...
[Regresión Logística con TF-IDF] ¡Entrenamiento finalizado con éxito!

🎯 Métricas del Modelo:
Accuracy (Exactitud): 0.4645

Reporte detallado de clasificación:
              precision    recall  f1-score   support

           0       0.56      0.64      0.60       410
           1       0.37      0.37      0.37       393
           2       0.35      0.34      0.34       395
           3       0.43      0.33      0.38       415
           4       0.57      0.64      0.61       387

    accuracy                           0.46      2000
   macro avg       0.46      0.47      0.46      2000
weighted avg       0.46      0.46      0.46      2000



In [7]:
from fastapi import FastAPI
from pydantic import BaseModel

# 1. Creamos la aplicación web
app = FastAPI()

# 2. Definimos cómo deben ser los datos que nos envían
class Comentario(BaseModel):
    texto: str

# 3. Ruta de bienvenida o estado (para saber que la API está viva)
@app.get("/")
def inicio():
    return {"mensaje": "¡Mi API de análisis de sentimiento está funcionando!"}

# 4. Ruta para predecir (recibe el texto del cliente y devuelve una respuesta simple)
@app.post("/predecir")
def predecir_sentimiento(datos: Comentario):
    texto_ingresado = datos.texto

    # Lógica súper sencilla basada en palabras clave
    if "malo" in texto_ingresado.lower() or "pésimo" in texto_ingresado.lower():
        resultado = "Negativo"
    elif "excelente" in texto_ingresado.lower() or "bueno" in texto_ingresado.lower():
        resultado = "Positivo"
    else:
        resultado = "Neutral"

    return {
        "comentario_recibido": texto_ingresado,
        "sentimiento_detectado": resultado
    }

In [11]:
from fastapi.testclient import TestClient

# Creamos el cliente de pruebas conectado a la app
cliente = TestClient(app)

# 1. Probamos la página de inicio
respuesta_raiz = cliente.get("/")
print("Respuesta de la raíz (/):", respuesta_raiz.json())

# 2. Probamos la API enviando un comentario de prueba
respuesta_prediccion = cliente.post("/predecir", json={"texto": "El producto me pareció excelente y de muy buena calidad"})
print("Respuesta de predicción (/predecir):", respuesta_prediccion.json())

Respuesta de la raíz (/): {'mensaje': '¡Mi API de análisis de sentimiento está funcionando!'}
Respuesta de predicción (/predecir): {'comentario_recibido': 'El producto me pareció excelente y de muy buena calidad', 'sentimiento_detectado': 'Positivo'}
